In [1]:
import pandas as pd
import numpy as np

In [2]:
hourly_raw = pd.read_csv("../data/openmeteo_donetsk_outputs/donetsk_hourly_raw.csv")
daily_raw = pd.read_csv("../data/openmeteo_donetsk_outputs/donetsk_daily_raw.csv")
df_weather = pd.read_csv("../data/weather_processed.csv")

In [3]:
pd.set_option("display.max_columns", None)

In [4]:
print(hourly_raw.shape)
print(daily_raw.shape)

(9120, 16)
(380, 3)


In [5]:
display(hourly_raw.head())
display(daily_raw.head())

,time,temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,snowfall,snow_depth,wind_gusts_10m,wind_speed_10m,wind_direction_10m,pressure_msl,cloud_cover,shortwave_radiation,uv_index,weather_code
0,2025-03-02T00:00,-5.6,-9.4,79,-8.7,0.0,0.0,0.02,10.4,4.4,145,1031.2,90,0.0,0.0,3
1,2025-03-02T01:00,-6.1,-9.9,80,-9.0,0.0,0.0,0.02,10.1,3.9,146,1030.9,83,0.0,0.0,3
2,2025-03-02T02:00,-6.5,-10.3,83,-8.9,0.0,0.0,0.02,9.4,3.9,146,1031.1,41,0.0,0.0,2
3,2025-03-02T03:00,-7.0,-10.6,85,-9.1,0.0,0.0,0.02,9.0,3.1,135,1030.3,70,0.0,0.0,2
4,2025-03-02T04:00,-7.3,-11.0,86,-9.3,0.0,0.0,0.02,7.2,3.1,126,1029.9,73,0.0,0.0,2


,time,sunrise,sunset
0,2025-03-02,2025-03-02T06:07,2025-03-02T17:14
1,2025-03-03,2025-03-03T06:05,2025-03-03T17:15
2,2025-03-04,2025-03-04T06:03,2025-03-04T17:17
3,2025-03-05,2025-03-05T06:01,2025-03-05T17:18
4,2025-03-06,2025-03-06T05:59,2025-03-06T17:20


In [6]:
hourly_raw.isna().sum()

time                    0
temperature_2m          0
apparent_temperature    0
relative_humidity_2m    0
dew_point_2m            0
precipitation           0
snowfall                0
snow_depth              0
wind_gusts_10m          0
wind_speed_10m          0
wind_direction_10m      0
pressure_msl            0
cloud_cover             0
shortwave_radiation     0
uv_index                0
weather_code            0
dtype: int64

In [7]:
hourly_raw["datetime_hour"] = pd.to_datetime(hourly_raw["time"])
hourly_raw["day_datetime"] = hourly_raw["datetime_hour"].dt.strftime("%Y-%m-%d")
hourly_raw["day_of_week"] = hourly_raw["datetime_hour"].dt.dayofweek
hourly_raw["hour"] = hourly_raw["datetime_hour"].dt.hour

daily_raw["day_datetime"] = pd.to_datetime(daily_raw["time"]).dt.strftime("%Y-%m-%d")
daily_raw["day_sunrise"] = pd.to_datetime(daily_raw["sunrise"]).dt.strftime("%H:%M:%S")
daily_raw["day_sunset"] = pd.to_datetime(daily_raw["sunset"]).dt.strftime("%H:%M:%S")

In [8]:
hourly = hourly_raw.rename(columns={
    "temperature_2m": "hour_temp",
    "apparent_temperature": "hour_feelslike",
    "relative_humidity_2m": "hour_humidity",
    "dew_point_2m": "hour_dew",
    "precipitation": "hour_precip",
    "snowfall": "hour_snow",
    "snow_depth": "hour_snowdepth",
    "wind_gusts_10m": "hour_windgust",
    "wind_speed_10m": "hour_windspeed",
    "wind_direction_10m": "hour_winddir",
    "pressure_msl": "hour_pressure",
    "cloud_cover": "hour_cloudcover",
    "shortwave_radiation": "hour_solarradiation",
    "uv_index": "hour_uvindex",
})

In [9]:
hourly["hour_solarenergy"] = hourly["hour_solarradiation"] * 0.0036 # hour_solarenergy приходить як потужність на одиницю площі,
                                                                    # а solarenergy це енергія за годину. 0.0036 це перерахунок за одну годину.

In [10]:
clear_codes = {0}
cloudy_codes = {1, 2, 3, 45, 48}
snow_codes = {71, 73, 75, 77, 85, 86}
rain_codes = {51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99}

hourly["hour_conditions_simple_Clear"] = hourly_raw["weather_code"].isin(clear_codes)
hourly["hour_conditions_simple_Cloudy"] = hourly_raw["weather_code"].isin(cloudy_codes)
hourly["hour_conditions_simple_Rain"] = hourly_raw["weather_code"].isin(rain_codes)
hourly["hour_conditions_simple_Snow"] = hourly_raw["weather_code"].isin(snow_codes)

In [11]:
daily_agg = (
    hourly.groupby("day_datetime", as_index=False)
    .agg(
        day_tempmax = ("hour_temp", "max"),
        day_tempmin = ("hour_temp", "min"),
        day_temp = ("hour_temp", "mean"),
        day_dew = ("hour_dew", "mean"),
        day_humidity = ("hour_humidity", "mean"),
        day_precip = ("hour_precip", "sum"),
        day_snow = ("hour_snow", "sum"),
        day_windgust= ("hour_windgust", "max"),
        day_cloudcover = ("hour_cloudcover", "mean"),
    )
)

In [12]:
precipcover = (
    hourly.assign(is_precip = (hourly["hour_precip"] > 0).astype(int))
    .groupby("day_datetime", as_index=False)["is_precip"].mean()
)

precipcover["day_precipcover"] = precipcover["is_precip"] * 100
precipcover = precipcover.drop(columns=["is_precip"])

In [13]:
daily_sun = daily_raw[["day_datetime", "day_sunrise", "day_sunset"]].copy()

daily_final = (
    daily_agg
    .merge(precipcover, on="day_datetime")
    .merge(daily_sun, on="day_datetime")
)

In [14]:
moonphase_map = df_weather[["day_datetime", "day_moonphase"]].copy()

moonphase_map["day_datetime"] = pd.to_datetime(moonphase_map["day_datetime"]).dt.strftime("%Y-%m-%d")
moonphase_map = moonphase_map.drop_duplicates(subset=["day_datetime"])

daily_final = daily_final.merge(moonphase_map, on="day_datetime")

In [15]:
donetsk_meta = (
    df_weather[df_weather["city_name"].astype(str).str.contains("Donetsk|Донець", case=False, na=False)][["city_name", "region_key", "region_id"]]
    .drop_duplicates()
)

display(donetsk_meta)

,city_name,region_key,region_id
3,Donetsk,Донецька,5


In [16]:
donetsk_meta_row = donetsk_meta.iloc[0]

In [17]:
donetsk_meta_row

city_name      Donetsk
region_key    Донецька
region_id            5
Name: 3, dtype: object

In [18]:
df_patch = hourly.merge(daily_final, on="day_datetime")

df_patch["city_name"] = donetsk_meta_row["city_name"]
df_patch["region_key"] = donetsk_meta_row["region_key"]
df_patch["region_id"] = donetsk_meta_row["region_id"]

In [19]:
final_cols = [
    "city_name", "datetime_hour", "day_tempmax", "day_tempmin", "day_temp", "day_dew", "day_humidity",
    "day_precip", "day_precipcover", "day_snow", "day_windgust", "day_cloudcover", "day_moonphase",
    "hour_temp", "hour_feelslike", "hour_humidity", "hour_dew", "hour_precip", "hour_snow", "hour_snowdepth",
    "hour_windgust", "hour_windspeed", "hour_winddir", "hour_pressure",
    "hour_cloudcover", "hour_solarradiation", "hour_solarenergy", "hour_uvindex",
    "day_of_week", "hour", "day_datetime", "day_sunrise", "day_sunset",
    "hour_conditions_simple_Clear", "hour_conditions_simple_Cloudy",
    "hour_conditions_simple_Rain", "hour_conditions_simple_Snow", "region_key", "region_id"
]

df_patch = df_patch[final_cols].sort_values("datetime_hour")

print(df_patch.shape)

(9120, 39)


In [20]:
display(df_patch.head())

,city_name,datetime_hour,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipcover,day_snow,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_of_week,hour,day_datetime,day_sunrise,day_sunset,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id
0,Donetsk,2025-03-02 00:00:00,1.7,-7.6,-2.829167,-8.604167,66.333333,0.0,0.0,0.0,22.7,84.333333,0.08,-5.6,-9.4,79,-8.7,0.0,0.0,0.02,10.4,4.4,145,1031.2,90,0.0,0.0,0.0,6,0,2025-03-02,06:07:00,17:14:00,False,True,False,False,Донецька,5
1,Donetsk,2025-03-02 01:00:00,1.7,-7.6,-2.829167,-8.604167,66.333333,0.0,0.0,0.0,22.7,84.333333,0.08,-6.1,-9.9,80,-9.0,0.0,0.0,0.02,10.1,3.9,146,1030.9,83,0.0,0.0,0.0,6,1,2025-03-02,06:07:00,17:14:00,False,True,False,False,Донецька,5
2,Donetsk,2025-03-02 02:00:00,1.7,-7.6,-2.829167,-8.604167,66.333333,0.0,0.0,0.0,22.7,84.333333,0.08,-6.5,-10.3,83,-8.9,0.0,0.0,0.02,9.4,3.9,146,1031.1,41,0.0,0.0,0.0,6,2,2025-03-02,06:07:00,17:14:00,False,True,False,False,Донецька,5
3,Donetsk,2025-03-02 03:00:00,1.7,-7.6,-2.829167,-8.604167,66.333333,0.0,0.0,0.0,22.7,84.333333,0.08,-7.0,-10.6,85,-9.1,0.0,0.0,0.02,9.0,3.1,135,1030.3,70,0.0,0.0,0.0,6,3,2025-03-02,06:07:00,17:14:00,False,True,False,False,Донецька,5
4,Donetsk,2025-03-02 04:00:00,1.7,-7.6,-2.829167,-8.604167,66.333333,0.0,0.0,0.0,22.7,84.333333,0.08,-7.3,-11.0,86,-9.3,0.0,0.0,0.02,7.2,3.1,126,1029.9,73,0.0,0.0,0.0,6,4,2025-03-02,06:07:00,17:14:00,False,True,False,False,Донецька,5


In [21]:
display(df_patch.tail())

,city_name,datetime_hour,day_tempmax,day_tempmin,day_temp,day_dew,day_humidity,day_precip,day_precipcover,day_snow,day_windgust,day_cloudcover,day_moonphase,hour_temp,hour_feelslike,hour_humidity,hour_dew,hour_precip,hour_snow,hour_snowdepth,hour_windgust,hour_windspeed,hour_winddir,hour_pressure,hour_cloudcover,hour_solarradiation,hour_solarenergy,hour_uvindex,day_of_week,hour,day_datetime,day_sunrise,day_sunset,hour_conditions_simple_Clear,hour_conditions_simple_Cloudy,hour_conditions_simple_Rain,hour_conditions_simple_Snow,region_key,region_id
9115,Donetsk,2026-03-16 19:00:00,9.8,2.7,5.7125,-1.041667,62.666667,0.1,4.166667,0.0,34.9,91.833333,0.92,6.6,2.9,67,0.9,0.0,0.0,0.0,34.6,11.8,70,1016.1,100,0.0,0.0,0.0,0,19,2026-03-16,05:39:00,17:34:00,False,True,False,False,Донецька,5
9116,Donetsk,2026-03-16 20:00:00,9.8,2.7,5.7125,-1.041667,62.666667,0.1,4.166667,0.0,34.9,91.833333,0.92,6.1,2.6,73,1.6,0.0,0.0,0.0,31.3,11.5,70,1017.0,100,0.0,0.0,0.0,0,20,2026-03-16,05:39:00,17:34:00,False,True,False,False,Донецька,5
9117,Donetsk,2026-03-16 21:00:00,9.8,2.7,5.7125,-1.041667,62.666667,0.1,4.166667,0.0,34.9,91.833333,0.92,5.7,2.3,76,1.8,0.0,0.0,0.0,29.5,10.7,70,1017.0,100,0.0,0.0,0.0,0,21,2026-03-16,05:39:00,17:34:00,False,True,False,False,Донецька,5
9118,Donetsk,2026-03-16 22:00:00,9.8,2.7,5.7125,-1.041667,62.666667,0.1,4.166667,0.0,34.9,91.833333,0.92,5.3,2.1,78,1.8,0.0,0.0,0.0,27.7,9.9,80,1017.4,100,0.0,0.0,0.0,0,22,2026-03-16,05:39:00,17:34:00,False,True,False,False,Донецька,5
9119,Donetsk,2026-03-16 23:00:00,9.8,2.7,5.7125,-1.041667,62.666667,0.1,4.166667,0.0,34.9,91.833333,0.92,5.1,1.9,79,1.8,0.1,0.0,0.0,26.3,9.4,83,1017.4,100,0.0,0.0,0.0,0,23,2026-03-16,05:39:00,17:34:00,False,False,True,False,Донецька,5


In [22]:
df_weather["datetime_hour"] = pd.to_datetime(df_weather["datetime_hour"])
df_weather["day_datetime"] = pd.to_datetime(df_weather["day_datetime"]).dt.strftime("%Y-%m-%d")

mask_old_tail = (
    df_weather["city_name"].astype(str).str.contains("Donetsk|Донець", case=False, na=False)
    & (df_weather["datetime_hour"] >= pd.Timestamp("2025-03-02 00:00:00"))
)

df_weather_patched = pd.concat(
    [
        df_weather.loc[~mask_old_tail],
        df_patch
    ],
    ignore_index=True
).sort_values(["region_id", "datetime_hour"]).reset_index(drop=True)

In [23]:
df_weather_patched = df_weather_patched.drop(
    df_weather_patched[
        (df_weather_patched["region_key"] == "Донецька") &
        (df_weather_patched["datetime_hour"] == "2025-03-30 03:00:00")
    ].index
) 

In [24]:
print("Duplicates:", df_weather_patched.duplicated(["region_id", "datetime_hour"]).sum())
print("Patch rows:", len(df_patch))
print("Final shape:", df_weather_patched.shape)

Duplicates: 0
Patch rows: 9120
Final shape: (853536, 39)


In [25]:
df_weather_patched.isna().sum()

city_name                        0
datetime_hour                    0
day_tempmax                      0
day_tempmin                      0
day_temp                         0
day_dew                          0
day_humidity                     0
day_precip                       0
day_precipcover                  0
day_snow                         0
day_windgust                     0
day_cloudcover                   0
day_moonphase                    0
hour_temp                        0
hour_feelslike                   0
hour_humidity                    0
hour_dew                         0
hour_precip                      0
hour_snow                        0
hour_snowdepth                   0
hour_windgust                    0
hour_windspeed                   0
hour_winddir                     0
hour_pressure                    0
hour_cloudcover                  0
hour_solarradiation              0
hour_solarenergy                 0
hour_uvindex                     0
day_of_week         

In [26]:
df_weather_patched.to_csv("final_weather.csv", index=False, encoding="utf-8-sig")